# Course 1 Lab — From DS Prototype to Production AI Service

## Scenario: Underwriting Guideline Assistant

You are leading the implementation of an enterprise assistant used by commercial-insurance underwriters.

The assistant must:

- answer questions from approved underwriting guidance;
- return document citations;
- avoid making unsupported decisions;
- remain testable without calling an LLM;
- allow infrastructure implementations to change without rewriting core logic;
- support future migration to OpenSearch, Bedrock Knowledge Bases, LiteLLM, AgentCore, identity-aware retrieval, and OpenTelemetry.

This lab focuses on **application architecture**, not retrieval SOTA. The local retriever is intentionally simple.

### What you will practice

`typing.Protocol` · Pydantic · dependency injection · async I/O · explicit errors · orchestration · logging · FastAPI · deterministic tests · optional Amazon Bedrock adapter

## 0. Optional environment setup

If your notebook environment does not already have the dependencies, run the next cell.

For a real repository, prefer a `pyproject.toml` and locked environment (for example with `uv`) rather than installing ad hoc from inside notebooks.

In [ ]:
# Uncomment if needed.
# %pip install -q "pydantic>=2" "fastapi>=0.115" "httpx>=0.27" "pytest>=8" "boto3>=1.35"

## 1. The prototype we want to grow out of

A DS prototype often combines retrieval, prompt construction, model invocation, and output formatting in one function.

That is useful for experimentation but creates problems when:

- the vector store changes;
- Bedrock is replaced by a gateway;
- authentication becomes mandatory;
- tests need to run without AWS;
- the API needs independent scaling;
- governance rules need to be inserted.

We will design explicit seams before adding infrastructure.

In [ ]:
def prototype_answer(question: str, documents: list[str]) -> str:
    # Deliberately simplistic prototype.
    matches = [d for d in documents if any(word.lower() in d.lower() for word in question.split())]
    context = "\n".join(matches[:2])
    return f"Prototype answer based on:\n{context}"

## 2. Domain models: represent the problem, not the provider

In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Protocol, Sequence, runtime_checkable
import asyncio
import logging
import re
import time
import uuid


@dataclass(frozen=True)
class Document:
    id: str
    title: str
    text: str
    source_uri: str
    allowed_groups: frozenset[str] = field(default_factory=lambda: frozenset({"underwriters"}))


@dataclass(frozen=True)
class RetrievedDocument:
    document: Document
    score: float


@dataclass(frozen=True)
class Citation:
    document_id: str
    title: str
    source_uri: str


@dataclass(frozen=True)
class Answer:
    text: str
    citations: tuple[Citation, ...]
    grounded: bool
    request_id: str

### Staff-level design question

Why are these models not Boto3 response objects or OpenSearch hits?

Because the application should speak its own domain language. Provider-specific formats belong in adapters.

## 3. Define ports/contracts with `Protocol`

In [ ]:
@runtime_checkable
class Retriever(Protocol):
    async def retrieve(
        self,
        query: str,
        *,
        k: int = 4,
        user_groups: frozenset[str] = frozenset({"underwriters"}),
    ) -> Sequence[RetrievedDocument]:
        ...


@runtime_checkable
class Generator(Protocol):
    async def generate(self, *, question: str, context: Sequence[RetrievedDocument]) -> str:
        ...

The application will depend on `Retriever` and `Generator`, not on OpenSearch, Bedrock, or LiteLLM.

That is dependency inversion in a practical AI application.

Later:

```text
Retriever -> OpenSearch adapter / Bedrock KB adapter
Generator -> Bedrock adapter / LiteLLM adapter
```

The orchestration service should not care which implementation is active.

## 4. Local deterministic retriever adapter

In [ ]:
def tokenize(text: str) -> set[str]:
    return set(re.findall(r"[a-z0-9]+", text.lower()))


class InMemoryRetriever:
    """Deterministic local adapter for architecture development and unit tests."""

    def __init__(self, documents: Sequence[Document], simulated_latency_s: float = 0.01):
        self._documents = tuple(documents)
        self._latency = simulated_latency_s

    async def retrieve(
        self,
        query: str,
        *,
        k: int = 4,
        user_groups: frozenset[str] = frozenset({"underwriters"}),
    ) -> Sequence[RetrievedDocument]:
        await asyncio.sleep(self._latency)  # simulate network/search I/O
        query_terms = tokenize(query)
        scored: list[RetrievedDocument] = []

        for doc in self._documents:
            # Defense-in-depth demo: filter before context generation.
            if doc.allowed_groups and doc.allowed_groups.isdisjoint(user_groups):
                continue

            doc_terms = tokenize(doc.text + " " + doc.title)
            if not query_terms:
                score = 0.0
            else:
                score = len(query_terms & doc_terms) / len(query_terms)

            if score > 0:
                scored.append(RetrievedDocument(document=doc, score=score))

        return sorted(scored, key=lambda x: x.score, reverse=True)[:k]

## 5. Build sample enterprise knowledge

In [ ]:
DOCUMENTS = [
    Document(
        id="UW-001",
        title="Commercial Property — Flood Exposure",
        text=(
            "Properties located in designated high-risk flood zones require a flood-risk review. "
            "The underwriter must verify flood mitigation measures and may require specialist approval "
            "before binding coverage."
        ),
        source_uri="s3://underwriting-guidelines/property/flood.md",
    ),
    Document(
        id="UW-002",
        title="Commercial Property — Roof Age",
        text=(
            "For buildings with roof systems older than 20 years, the underwriter should request "
            "evidence of inspection, maintenance, or replacement. Exceptions require documented rationale."
        ),
        source_uri="s3://underwriting-guidelines/property/roof-age.md",
    ),
    Document(
        id="UW-003",
        title="General Underwriting Authority",
        text=(
            "The AI assistant may provide guidance and source references but must not bind coverage, "
            "approve exceptions, or replace the accountable underwriter's decision."
        ),
        source_uri="s3://underwriting-guidelines/governance/authority.md",
    ),
    Document(
        id="SEC-001",
        title="Restricted Executive Exception Process",
        text="Executive exceptions require restricted approval and confidential supporting documentation.",
        source_uri="s3://underwriting-guidelines/restricted/exceptions.md",
        allowed_groups=frozenset({"underwriting-managers"}),
    ),
]

## 6. Deterministic generator adapter

In [ ]:
class DeterministicGroundedGenerator:
    """A fake generator that makes orchestration tests deterministic."""

    async def generate(self, *, question: str, context: Sequence[RetrievedDocument]) -> str:
        await asyncio.sleep(0.01)
        if not context:
            return "I could not find approved guidance that supports an answer."

        bullets = " ".join(
            f"[{item.document.id}] {item.document.text}"
            for item in context[:2]
        )
        return (
            f"Based on the approved guidance: {bullets} "
            "The underwriter remains responsible for the final decision."
        )

## 7. Explicit application errors

In [ ]:
class ApplicationError(Exception):
    """Base error safe for application-layer translation."""


class RetrievalError(ApplicationError):
    pass


class GenerationError(ApplicationError):
    pass


class DependencyTimeout(ApplicationError):
    pass

## 8. Application orchestration service

In [ ]:
logger = logging.getLogger("underwriting_ai")
if not logger.handlers:
    handler = logging.StreamHandler()
    formatter = logging.Formatter(
        "%(asctime)s %(levelname)s %(name)s %(message)s"
    )
    handler.setFormatter(formatter)
    logger.addHandler(handler)
logger.setLevel(logging.INFO)


class UnderwritingAssistantService:
    def __init__(
        self,
        *,
        retriever: Retriever,
        generator: Generator,
        retrieval_k: int = 4,
        timeout_s: float = 4.0,
    ):
        self._retriever = retriever
        self._generator = generator
        self._retrieval_k = retrieval_k
        self._timeout_s = timeout_s

    async def answer(
        self,
        question: str,
        *,
        user_groups: frozenset[str] = frozenset({"underwriters"}),
    ) -> Answer:
        request_id = str(uuid.uuid4())
        started = time.perf_counter()

        if not question.strip():
            raise ValueError("question must not be empty")

        try:
            async with asyncio.timeout(self._timeout_s):
                try:
                    retrieved = await self._retriever.retrieve(
                        question,
                        k=self._retrieval_k,
                        user_groups=user_groups,
                    )
                except Exception as exc:
                    raise RetrievalError("Retrieval dependency failed") from exc

                # Fail closed on grounding: no context -> no invented answer.
                if not retrieved:
                    text = "I could not find approved guidance that supports an answer."
                    citations: tuple[Citation, ...] = ()
                    grounded = False
                else:
                    try:
                        text = await self._generator.generate(
                            question=question,
                            context=retrieved,
                        )
                    except Exception as exc:
                        raise GenerationError("Generation dependency failed") from exc

                    citations = tuple(
                        Citation(
                            document_id=item.document.id,
                            title=item.document.title,
                            source_uri=item.document.source_uri,
                        )
                        for item in retrieved
                    )
                    grounded = True

        except TimeoutError as exc:
            raise DependencyTimeout(
                f"Request exceeded {self._timeout_s:.1f}s application timeout"
            ) from exc

        elapsed_ms = (time.perf_counter() - started) * 1000
        logger.info(
            "request_completed request_id=%s grounded=%s citations=%d latency_ms=%.1f",
            request_id,
            grounded,
            len(citations),
            elapsed_ms,
        )

        return Answer(
            text=text,
            citations=citations,
            grounded=grounded,
            request_id=request_id,
        )

### Notice what the service does *not* know

It does not know:

- Boto3;
- AWS credentials;
- OpenSearch request syntax;
- FastAPI;
- HTTP status codes;
- LiteLLM;
- Terraform;
- ECS.

That separation is intentional.

## 9. Run the service locally

In [ ]:
retriever = InMemoryRetriever(DOCUMENTS)
generator = DeterministicGroundedGenerator()
service = UnderwritingAssistantService(
    retriever=retriever,
    generator=generator,
    retrieval_k=3,
    timeout_s=2.0,
)

answer = await service.answer(
    "What should I do if the building roof is more than 20 years old?"
)

print(answer.text)
print("\nCitations:")
for citation in answer.citations:
    print("-", citation)

## 10. Verify access-aware retrieval

In [ ]:
normal_user = await service.answer(
    "What is the executive exception process?",
    user_groups=frozenset({"underwriters"}),
)

manager = await service.answer(
    "What is the executive exception process?",
    user_groups=frozenset({"underwriting-managers"}),
)

print("Underwriter:")
print(normal_user.text)
print("\nManager:")
print(manager.text)
print("\nManager citations:", [c.document_id for c in manager.citations])

## 11. Structured concurrency example: query independent sources in parallel

Suppose approved guidelines and product manuals are separate retrieval systems.

If both are independent I/O calls, they can potentially run concurrently.

`TaskGroup` gives us structured lifetime and failure behavior.

In [ ]:
class FederatedRetriever:
    def __init__(self, retrievers: Sequence[Retriever]):
        self._retrievers = tuple(retrievers)

    async def retrieve(
        self,
        query: str,
        *,
        k: int = 4,
        user_groups: frozenset[str] = frozenset({"underwriters"}),
    ) -> Sequence[RetrievedDocument]:
        tasks = []
        async with asyncio.TaskGroup() as tg:
            for retriever in self._retrievers:
                tasks.append(
                    tg.create_task(
                        retriever.retrieve(query, k=k, user_groups=user_groups)
                    )
                )

        merged = [item for task in tasks for item in task.result()]

        # Real systems need a deliberate score-normalization/fusion strategy.
        deduped: dict[str, RetrievedDocument] = {}
        for item in merged:
            current = deduped.get(item.document.id)
            if current is None or item.score > current.score:
                deduped[item.document.id] = item

        return sorted(deduped.values(), key=lambda x: x.score, reverse=True)[:k]

In [ ]:
product_docs = [
    Document(
        id="PROD-101",
        title="Property Plus Product Manual",
        text="Property Plus accepts commercial property submissions subject to underwriting authority and required risk reviews.",
        source_uri="s3://product-manuals/property-plus.md",
    )
]

federated = FederatedRetriever([
    InMemoryRetriever(DOCUMENTS, simulated_latency_s=0.05),
    InMemoryRetriever(product_docs, simulated_latency_s=0.05),
])

federated_results = await federated.retrieve("commercial property underwriting")
[(r.document.id, round(r.score, 2)) for r in federated_results]

### Architecture discussion

Parallelism is not automatically correct.

Ask:

- Are the calls independent?
- Is one source authoritative?
- What should happen if one retriever fails?
- Are scores comparable?
- Should partial retrieval be allowed?
- What is the latency budget?
- Will concurrency overload a dependency?

You should be able to make these decisions rather than simply using `asyncio.gather`.

## 12. Pydantic at the API boundary

In [ ]:
from pydantic import BaseModel, Field


class AskRequest(BaseModel):
    question: str = Field(min_length=3, max_length=2000)
    user_groups: list[str] = Field(default_factory=lambda: ["underwriters"])


class CitationResponse(BaseModel):
    document_id: str
    title: str
    source_uri: str


class AskResponse(BaseModel):
    answer: str
    grounded: bool
    request_id: str
    citations: list[CitationResponse]

## 13. FastAPI as a thin HTTP adapter

In [ ]:
from fastapi import Depends, FastAPI, HTTPException


def build_service() -> UnderwritingAssistantService:
    # In a real app, construct long-lived clients at app startup / lifespan.
    return UnderwritingAssistantService(
        retriever=InMemoryRetriever(DOCUMENTS),
        generator=DeterministicGroundedGenerator(),
        retrieval_k=3,
        timeout_s=2.0,
    )


app = FastAPI(title="Underwriting AI")

# Simple demo singleton. Later modules will cover lifecycle, clients,
# credentials, deployment and observability.
_default_service = build_service()


def get_service() -> UnderwritingAssistantService:
    return _default_service


@app.post("/ask", response_model=AskResponse)
async def ask(
    request: AskRequest,
    assistant: UnderwritingAssistantService = Depends(get_service),
) -> AskResponse:
    try:
        result = await assistant.answer(
            request.question,
            user_groups=frozenset(request.user_groups),
        )
    except DependencyTimeout as exc:
        raise HTTPException(status_code=504, detail=str(exc)) from exc
    except ApplicationError as exc:
        # Do not leak provider-specific internals.
        raise HTTPException(status_code=503, detail=str(exc)) from exc

    return AskResponse(
        answer=result.text,
        grounded=result.grounded,
        request_id=result.request_id,
        citations=[
            CitationResponse(
                document_id=c.document_id,
                title=c.title,
                source_uri=c.source_uri,
            )
            for c in result.citations
        ],
    )


print("FastAPI app created. In a normal Python module you could run it with a FastAPI/Uvicorn server.")

## 14. Test through the HTTP boundary

In [ ]:
from fastapi.testclient import TestClient

client = TestClient(app)

response = client.post(
    "/ask",
    json={
        "question": "What is required for an old roof?",
        "user_groups": ["underwriters"],
    },
)

print("Status:", response.status_code)
print(response.json())

assert response.status_code == 200
assert response.json()["citations"]

## 15. Deterministic unit tests without AWS

In [ ]:
class RecordingGenerator:
    def __init__(self):
        self.calls = []

    async def generate(self, *, question: str, context: Sequence[RetrievedDocument]) -> str:
        self.calls.append((question, context))
        return "Controlled response"


async def test_service_uses_retrieved_context():
    retriever = InMemoryRetriever(DOCUMENTS, simulated_latency_s=0)
    generator = RecordingGenerator()
    test_service = UnderwritingAssistantService(
        retriever=retriever,
        generator=generator,
        timeout_s=1,
    )

    result = await test_service.answer("roof older than 20 years")

    assert result.grounded is True
    assert len(result.citations) >= 1
    assert len(generator.calls) == 1
    assert any(
        item.document.id == "UW-002"
        for item in generator.calls[0][1]
    )


async def test_no_context_does_not_call_generator():
    retriever = InMemoryRetriever(DOCUMENTS, simulated_latency_s=0)
    generator = RecordingGenerator()
    test_service = UnderwritingAssistantService(
        retriever=retriever,
        generator=generator,
        timeout_s=1,
    )

    result = await test_service.answer("quantum submarine warranty")

    assert result.grounded is False
    assert generator.calls == []


await test_service_uses_retrieved_context()
await test_no_context_does_not_call_generator()
print("Deterministic service tests passed.")

## 16. Test timeout behavior

In [ ]:
slow_service = UnderwritingAssistantService(
    retriever=InMemoryRetriever(DOCUMENTS, simulated_latency_s=0.2),
    generator=DeterministicGroundedGenerator(),
    timeout_s=0.05,
)

try:
    await slow_service.answer("roof age")
except DependencyTimeout as exc:
    print("Expected timeout:", exc)
else:
    raise AssertionError("Expected DependencyTimeout")

## 17. Optional: Amazon Bedrock generator adapter

This cell defines an adapter but does **not** make a request by default.

### Prerequisites

- AWS credentials resolved using your organization's approved mechanism;
- permission to invoke the chosen Bedrock model;
- a supported model/inference profile ID;
- `boto3` installed.

For current Bedrock API recommendations, consult AWS documentation because the API surface and supported models evolve.

The key architectural lesson is that Boto3 stays inside the adapter.

In [ ]:
class BedrockConverseGenerator:
    def __init__(
        self,
        *,
        model_id: str,
        region_name: str = "us-west-2",
        max_tokens: int = 700,
        temperature: float = 0.0,
    ):
        import boto3

        self._client = boto3.client("bedrock-runtime", region_name=region_name)
        self._model_id = model_id
        self._max_tokens = max_tokens
        self._temperature = temperature

    def _converse_sync(self, *, question: str, context: Sequence[RetrievedDocument]) -> str:
        context_text = "\n\n".join(
            f"[{item.document.id}] {item.document.text}"
            for item in context
        )

        response = self._client.converse(
            modelId=self._model_id,
            system=[
                {
                    "text": (
                        "You are an underwriting guidance assistant. "
                        "Use only the supplied context. Cite supporting document IDs. "
                        "If the context is insufficient, say so. "
                        "Never make or approve an underwriting decision."
                    )
                }
            ],
            messages=[
                {
                    "role": "user",
                    "content": [
                        {
                            "text": f"Question: {question}\n\nApproved context:\n{context_text}"
                        }
                    ],
                }
            ],
            inferenceConfig={
                "maxTokens": self._max_tokens,
                "temperature": self._temperature,
            },
        )

        blocks = response["output"]["message"]["content"]
        return "".join(block.get("text", "") for block in blocks)

    async def generate(self, *, question: str, context: Sequence[RetrievedDocument]) -> str:
        # boto3's standard client call is synchronous. Running it directly inside an async
        # route would block the event loop. For this learning lab we offload it to a thread.
        # In production, evaluate client behavior, throughput, cancellation and connection
        # management deliberately rather than applying this pattern blindly.
        return await asyncio.to_thread(
            self._converse_sync,
            question=question,
            context=context,
        )

### Optional invocation

Replace the placeholder with a model or inference profile that your AWS account is allowed to invoke.

```python
bedrock_generator = BedrockConverseGenerator(
    model_id="YOUR_MODEL_OR_INFERENCE_PROFILE_ID",
    region_name="YOUR_REGION",
)

bedrock_service = UnderwritingAssistantService(
    retriever=InMemoryRetriever(DOCUMENTS),
    generator=bedrock_generator,
)

result = await bedrock_service.answer("What is required for a roof older than 20 years?")
print(result)
```

Notice: the `UnderwritingAssistantService` did not change.

## 18. Failure-injection exercise

In [ ]:
class FailingRetriever:
    async def retrieve(
        self,
        query: str,
        *,
        k: int = 4,
        user_groups: frozenset[str] = frozenset({"underwriters"}),
    ) -> Sequence[RetrievedDocument]:
        raise ConnectionError("simulated search outage")


failure_service = UnderwritingAssistantService(
    retriever=FailingRetriever(),
    generator=DeterministicGroundedGenerator(),
)

try:
    await failure_service.answer("roof age")
except RetrievalError as exc:
    print("Application translated infrastructure failure to:", type(exc).__name__, "-", exc)

## 19. Your Staff-level exercises

### A. Partial retrieval

Change `FederatedRetriever` so that one source may fail while another succeeds.

Before coding, write a decision note:

- When is partial knowledge safe?
- Should the user be told?
- What telemetry is required?
- Which domains should fail closed?

### B. Reranker

Add:

```python
class Reranker(Protocol):
    async def rerank(...)
```

Where should it be injected?

Should the service know the reranker implementation?

### C. Model fallback

Create:

```text
PrimaryGenerator
       |
failure?
       v
FallbackGenerator
```

Should fallback live in:

- the RAG service;
- a generator decorator;
- a future LiteLLM/model gateway?

Defend your answer.

### D. API versioning

Change the external `AskResponse` schema without changing the internal `Answer` domain object.

What does this teach you about boundary models?

### E. Security

The notebook filters `allowed_groups` in the retriever.

Explain why an enterprise implementation should not rely only on post-retrieval application filtering.

### F. Code review

Review this proposal:

```python
@app.post("/ask")
async def ask(question: str):
    bedrock = boto3.client("bedrock-runtime")
    docs = opensearch.search(question)
    return bedrock.invoke_model(prompt=question + str(docs))
```

Produce a written review covering:

- architecture;
- coupling;
- resource lifecycle;
- async/blocking behavior;
- validation;
- security;
- testability;
- failure handling;
- observability readiness;
- migration path.

This is the most important exercise in the notebook.

## 20. Architecture checkpoint

At this point, your mental model should be:

```text
                          HTTP
                           |
                    Pydantic boundary
                           |
                        FastAPI
                           |
                 Application orchestration
                     /              \
                    /                \
          Retriever Protocol      Generator Protocol
                 |                      |
          Local / Search             Local / Bedrock
             adapters                  adapters
```

Later courses will extend this with:

```text
Entra ID / SSO
IAM / Agent Identity
OpenSearch / Bedrock KB
LiteLLM model gateway
AgentCore
OpenTelemetry
CloudWatch
Kinesis / SQS / EventBridge
RAG + Agent evaluation
AI red-team gates
GitHub Actions
Terraform
ECS / EKS / Lambda
```

The application core should not collapse when those are introduced.

## 21. Self-assessment

You are finished only when you can explain **why** each boundary exists.

Score yourself from 1–5:

| Skill | Score |
|---|---:|
| Modern Python project structure | |
| Typing and Protocols | |
| Pydantic boundary validation | |
| Dependency injection | |
| Async I/O | |
| Timeouts and failure translation | |
| FastAPI architecture | |
| Unit vs integration testing | |
| Bedrock adapter design | |
| Architecture/code review | |

### Exit question

A team wants to replace direct Bedrock calls with LiteLLM next quarter.

**Which files/components should change, and which should remain unchanged?**

If your answer is "most of the application", the architecture is too coupled.